# <center>TensorRT Plugin 加速推理</center>

NVIDIA TensorRT 通过不断扩展支持的网络层类型来适应深度学习模型的快速迭代。但在实际应用中，开发者可能遇到现有算子无法满足需求的情况（如特殊算子优化、硬件特性适配等）。此时，**自定义插件（Plugin）** 成为扩展 TensorRT 功能的关键技术。TensorRT 提供了丰富的[标准插件库](https://github.com/NVIDIA/TensorRT/tree/main/plugin#tensorrt-plugins)，同时支持开发者编写自定义插件。

为了在应用程序中使用预置插件，需要在代码中调用 `initLibNvInferPlugins` 函数并链接 `nvinfer_plugin` 动态库。如果现有的插件无法满足特定需求，则可以选择编写并集成自定义插件，以满足特定的功能要求。

## 如何实现自定义插件

为了确保 TensorRT 能够正确识别并使用自定义插件，需要遵循以下步骤：

1. **实现插件类**：创建一个插件类，定义其功能和行为。从 TensorRT 10.0 版本开始，推荐使用 `IPluginV3` 接口，其他接口已被弃用。`IPluginV3` 是一组功能接口的封装，定义了三种功能：核心（Core）、构建（Build）和运行时（Runtime）。具体实现时，需要根据插件需求实现 `IPluginV3OneCore`、`IPluginV3OneBuild` 和 `IPluginV3OneRuntime` 等接口。
2. **实现插件创建器类**：设计一个插件创建器类，用于生成插件实例。插件创建器类需要继承自 `IPluginCreatorV3One`，并实现 `createPlugin` 方法。该方法用于根据输入的参数创建插件对象。
3. **注册插件创建器**：将插件创建器注册到 TensorRT 的插件注册表中，以便 TensorRT 能够识别和加载自定义插件。可以通过静态注册（使用 `REGISTER_TENSORRT_PLUGIN` 宏）或动态注册（通过 `registerCreator` 方法）来完成。
4. **将插件添加到 TensorRT 网络**：通过 TensorRT API 或 ONNX API 将插件实例添加到网络中。

[MatrixAddPlugin](./course_codes/01.04.MatrixAddPlugin) 是一个简单的自定义插件实现示例，该插件实现了矩阵相加功能。

## 如何添加/替换插件

在实际应用中，可能需要将自定义插件替换现有的 TensorRT 层或添加到网络中。以下是一个完整的流程示例，包括使用 PyTorch 实现矩阵相加模块、导出为 ONNX 模型，并通过 ONNX-GraphSurgeon 修改模型以使用自定义插件。

In [ ]:
import torch

class MatrixAdd(torch.nn.Module):
    def __init__(self):
        super(MatrixAdd, self).__init__()

    def forward(self, input1, input2):
        return input1 + input2

def export2onnx(model, save_path):
    # 创建输入张量
    input_tensor1 = torch.randn(3, 3, dtype=torch.float32)
    input_tensor2 = torch.randn(3, 3, dtype=torch.float32)
 
    # 导出为ONNX格式
    torch.onnx.export(model, (input_tensor1, input_tensor2), save_path, opset_version=11, 
                      input_names=['input1', 'input2'],output_names=['output'],)

model = MatrixAdd()
save_path = "./course_data/models/MATRIX-ADD/matrix_add.onnx"
export2onnx(model, save_path)

In [ ]:
import netron

address = netron.start(save_path, verbosity=0, browse=False)
netron.widget(address)

接下来，我们将在PyTorch中注册一个自定义算子，用于替换原生的Add节点。

In [ ]:
class MatrixAddOperator(torch.autograd.Function):

    @staticmethod
    def forward(ctx, input1: torch.Tensor, input2: torch.Tensor, plugin_version: str = '1') -> torch.Tensor:
        return input1 + input2

    @staticmethod
    def symbolic(g, input1, input2, plugin_version='1') -> torch.Value:
        return g.op("TRT::MatrixAdd", input1, input2, outputs=1, plugin_version_s=plugin_version)

class MatrixAdd4Torch(torch.nn.Module):
    def __init__(self):
        super(MatrixAdd4Torch, self).__init__()
        self.matrix_add = MatrixAddOperator.apply

    def forward(self, input1, input2):
        return self.matrix_add(input1, input2)

model = MatrixAdd4Torch()
save_path = "./course_data/models/MATRIX-ADD/matrix_add-plugin-torch.onnx"
export2onnx(model, save_path)

In [ ]:
address = netron.start(save_path, verbosity=0, browse=False)
netron.widget(address)

如果导出的ONNX模型的输出维度与期望的不符，我们可以手动修改onnx设置为正确的输出维度。

In [ ]:
import onnx
import onnxsim

model_path = save_path  # 模型的路径
output_shape = {'output': [3, 3]}  # 指定输出形状

# 加载ONNX模型
model_onnx = onnx.load(model_path)
onnx.checker.check_model(model_onnx)  # 检查模型的有效性

# 修改模型的输出形状
for node in model_onnx.graph.output:
    for idx, dim in enumerate(node.type.tensor_type.shape.dim):
        dim.dim_param = str(output_shape[node.name][idx])

# 简化ONNX模型
model_onnx, check = onnxsim.simplify(model_onnx)
assert check, "Simplified ONNX model could not be validated"  # 确保简化后的模型有效

# 保存简化后的ONNX模型
simplified_model_path = model_path.replace(".onnx", "_sim.onnx")
onnx.save(model_onnx, simplified_model_path)

In [ ]:
address = netron.start(simplified_model_path, verbosity=0, browse=False)
netron.widget(address)

最后，我们将学习使用 [onnx-graphsurgeon](https://github.com/NVIDIA/TensorRT/tree/main/tools/onnx-graphsurgeon) 库来修改 ONNX 模型，将原生的 Add 节点替换为自定义的操作。有关 onnx-graphsurgeon 的使用示例可参考 [onnx-graphsurgeon/examples](https://github.com/NVIDIA/TensorRT/tree/main/tools/onnx-graphsurgeon/examples)，这里将不再赘述。

这里给出了两种将原生的 Add 节点替换为自定义的插件的方法，一种是找出需要替换的节点，修改节点的操作类型与属性。另一种方法是使用 `Graph.layer()` 来实现插件的定义，并使用 `Graph.register()` 进行注册。

In [ ]:
import numpy as np
import onnx_graphsurgeon as gs

# 注册自定义操作MatrixAdd到onnx_graphsurgeon的Graph类中
@gs.Graph.register(opsets=[11])
def matrix_add(self, a, b):
    # 创建一个自定义的MatrixAdd层
    return self.layer(
        op="MatrixAdd", name="MatrixAdd",
        attrs={"plugin_version": "1"}, inputs=[a, b],
        outputs=[gs.Variable(name="output", dtype=np.float32, shape=(3, 3))]
    )

# 获取模型的图
model_path = "./course_data/models/MATRIX-ADD/matrix_add.onnx"
save_model = "./course_data/models/MATRIX-ADD/matrix_add-plugin-graphsurgeon.onnx"
graph = gs.import_onnx(onnx.load(model_path))

# 方法1：直接使用自定义的MatrixAdd函数替换所有输出节点
# graph.outputs = graph.matrix_add(graph.inputs[0], graph.inputs[1])

# 方法2：找到需要修改的节点
add_node = [node for node in graph.nodes if node.op == "Add" and len(node.inputs) == 2 and node.outputs[0].name == "output"][0]
# 将 Add 节点替换为 MatrixAdd
add_node.op, add_node.name, add_node.attrs["plugin_version"] = "MatrixAdd", "MatrixAdd_nodec", "1"

# 清理和排序图
graph.cleanup(remove_unused_graph_inputs=True).toposort()
# 推断图的形状
model = onnx.shape_inference.infer_shapes(gs.export_onnx(graph))
# 保存修改后的模型
onnx.save(model, save_model)

In [ ]:
address = netron.start(save_model, verbosity=0, browse=False)
netron.widget(address)

## 推理带自定义插件的引擎

In [ ]:
command = (
    f"/usr/src/tensorrt/bin/trtexec --onnx={save_model} --saveEngine={save_model.replace('.onnx', '.engine')}"
    " --staticPlugins=./course_codes/01.04.MatrixAddPlugin/libs/libMatrixAddPlugin.so"
    " --setPluginsToSerialize=./course_codes/01.04.MatrixAddPlugin/libs/libMatrixAddPlugin.so"
)

from course_functions.command_runner import run_command
run_command(command)

In [ ]:
import numpy as np
from course_functions.inference import BaseModel

class MatrixAddModel(BaseModel):
    def __init__(self, engine_file: str) -> None:
        # 初始化基类，加载TensorRT引擎
        super().__init__(engine_file)
        # 确保有3个tensor（2个输入，1个输出）
        assert len(self.tensors) == 3, "MatrixAddModel expects the engine to have exactly 3 tensors (2 inputs and 1 output)."

    def preprocess(self, data1: np.ndarray, data2: np.ndarray) -> None:
        """前处理：将输入数据复制到模型的输入张量"""
        # 确保输入数据的形状与模型的输入张量形状相匹配
        assert data1.shape == self.tensors[0].shape, "Data1 shape does not match input tensor shape."
        assert data2.shape == self.tensors[1].shape, "Data2 shape does not match input tensor shape."
        # 将数据复制到对应的输入张量
        self.tensors[0].memory.host = data1
        self.tensors[1].memory.host = data2

    def postprocess(self) -> np.ndarray:
        """后处理：从模型的输出张量获取结果"""
        # 将输出张量的数据复制到numpy数组，并重新整形为原始形状
        result = self.tensors[2].memory.host[: np.prod(self.tensors[2].shape)].reshape(self.tensors[2].shape)
        return result

    def predict(self, data1: np.ndarray, data2: np.ndarray) -> np.ndarray:
        """推理预测：执行模型的前处理、推理和后处理"""
        self.preprocess(data1, data2)
        self.inference()
        return self.postprocess()

In [ ]:
# 初始化模型
model = MatrixAddModel("./course_data/models/MATRIX-ADD/matrix_add-plugin-graphsurgeon.engine")
model.warmup()

# 生成随机输入张量
input_tensor1 = np.random.randn(3, 3).astype(np.float32)
input_tensor2 = np.random.randn(3, 3).astype(np.float32)

# 预测结果
result = model.predict(input_tensor1, input_tensor2)
print(f"result: \n{result}")

# 计算实际结果
true_data = input_tensor1 + input_tensor2
print(f"true_data: \n{true_data}")

# 比较模型结果和实际结果
if np.allclose(result, true_data, atol=1e-5):
    print("Model result is accurate.")
else:
    print("Model result is inaccurate.")

## <center>总结</center>

通过本节课的学习，我们掌握了 TensorRT 自定义插件的开发与集成，以扩展其功能并加速推理。自定义插件的实现需要遵循一定的规范，包括插件类和插件创建器类的实现，以及插件的注册和使用。此外，通过在 PyTorch 中定义自定义操作，并将其注册为自定义算子或者通过 ONNX-GraphSurgeon 修改模型以使用自定义插件是一种常见的方法。希望本课件能够帮助您更好地理解和应用 TensorRT 自定义插件的开发。